# P-702 – Hourly Energy Consumption Forecasting

## Integrated EDA, Train/Test Split, Five-Model Comparison and Final 30-Day Forecast

### Project objective
Build an hourly power-consumption forecasting solution for the PJM dataset.

### Required outcomes
1. Perform exploratory data analysis (EDA) to understand hourly, weekly, seasonal, holiday and long-term patterns.
2. Prepare the time series and handle duplicate/missing hourly timestamps.
3. Split the data chronologically, using the last year as the test set.
4. Build and compare five forecasting approaches.
5. Select the best model using MAE, with RMSE and MAPE shown for confirmation.
6. Generate the next 30 days of hourly forecasts (720 hours).
7. Export the final hourly forecast and daily summary.

> **Important:** This notebook combines the EDA approach from the EDA notebook with the chronological train/test and five-model forecasting approach from the model-comparison notebook. The final forecasting logic follows the model-comparison notebook, where Previous-Day Seasonal Naive (24h) was selected based on the lowest test-set MAE.


## 1. Environment and imports

The notebook uses standard Python data-science libraries plus XGBoost and Statsmodels for the five-model comparison and time-series decomposition.

In [ ]:
# If required, install the packages once in your environment.
# Uncomment the next line only if these packages are not already installed.
# %pip install pandas numpy matplotlib seaborn scikit-learn xgboost statsmodels openpyxl

import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor

from xgboost import XGBRegressor
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

RANDOM_STATE = 42
FORECAST_DAYS = 30
FORECAST_HOURS = FORECAST_DAYS * 24

print("Environment ready.")


## 2. Load the PJM dataset

In [ ]:
# ---------------------------------------------------------
# Dataset discovery
# ---------------------------------------------------------
DATA_PATH = r"D:\Richa\Data Science\Project\PJMW_MW_Hourly.xlsx"

candidate_names = [
    "PJMW_hourly.csv",
    "PJMW_MW_Hourly.xlsx",
    "PJMW_hourly.xlsx",
]

if DATA_PATH is None:
    search_roots = [os.getcwd()]
    for root in search_roots:
        for name in candidate_names:
            direct = os.path.join(root, name)
            if os.path.exists(direct):
                DATA_PATH = direct
                break
        if DATA_PATH:
            break

if DATA_PATH is None:
    found = []
    for pattern in ["**/PJMW_hourly.csv", "**/PJMW_MW_Hourly.xlsx", "**/PJMW_hourly.xlsx"]:
        found.extend(glob.glob(os.path.join(os.getcwd(), pattern), recursive=True))
    if found:
        DATA_PATH = found[0]

if DATA_PATH is None:
    raise FileNotFoundError(
        "PJMW dataset was not found. Put PJMW_hourly.csv or PJMW_MW_Hourly.xlsx "
        "in the notebook folder, or set DATA_PATH manually."
    )

print("Using dataset:", DATA_PATH)

# Load CSV or Excel depending on file extension.
if DATA_PATH.lower().endswith(".csv"):
    raw_df = pd.read_csv(DATA_PATH)
else:
    # The source ModelBuilding notebook used sheet_name='PJMW_hourly'.
    try:
        raw_df = pd.read_excel(DATA_PATH, sheet_name="PJMW_hourly")
    except Exception:
        raw_df = pd.read_excel(DATA_PATH)

# Standardize the expected columns.
raw_df["Datetime"] = pd.to_datetime(raw_df["Datetime"])
raw_df = raw_df[["Datetime", "PJMW_MW"]].copy()
raw_df = raw_df.sort_values("Datetime").reset_index(drop=True)

print("Shape:", raw_df.shape)
display(raw_df.head())
display(raw_df.describe())


## 3. Data quality and time-series preparation

In [ ]:
# Data quality checks before regularization
print("Missing target values:", raw_df["PJMW_MW"].isna().sum())
print("Duplicate timestamps:", raw_df["Datetime"].duplicated().sum())
print("Date range:", raw_df["Datetime"].min(), "to", raw_df["Datetime"].max())

# Average duplicate timestamps, then enforce a continuous hourly index.
ts = (
    raw_df.groupby("Datetime")["PJMW_MW"]
          .mean()
          .sort_index()
          .asfreq("h")
)

missing_after_regularization = int(ts.isna().sum())

# Fill genuine missing hourly observations using time interpolation.
ts = ts.interpolate(method="time").ffill().bfill()

print("Regular hourly observations:", len(ts))
print("Missing observations filled:", missing_after_regularization)
print("Final date range:", ts.index.min(), "to", ts.index.max())


## 4. Exploratory Data Analysis (EDA)
- overall distribution,
- hourly pattern,
- day-of-week pattern,
- monthly/seasonal pattern,
- seasonal hourly pattern,
- weekday vs weekend,
- holidays,
- long-term trend,
- daily trend,
- outliers,
- correlation,
- time-series decomposition.

In [ ]:
# Create an EDA dataframe from the cleaned hourly series.
eda = ts.to_frame("PJMW_MW").copy()

eda["Hour"] = eda.index.hour
eda["DayOfWeek"] = eda.index.dayofweek
eda["Month"] = eda.index.month
eda["Year"] = eda.index.year
eda["DayOfYear"] = eda.index.dayofyear
eda["Is_Weekend"] = (eda["DayOfWeek"] >= 5).astype(int)

# US federal holidays, matching the source EDA approach.
holiday_calendar = USFederalHolidayCalendar()
holiday_index = holiday_calendar.holidays(
    start=eda.index.min().date(),
    end=eda.index.max().date()
)
holiday_dates = set(pd.to_datetime(holiday_index).date)

eda["Is_Holiday"] = [d.date() in holiday_dates for d in eda.index]

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    return "Autumn"

eda["Season"] = eda["Month"].apply(get_season)

print("Overall mean MW:", round(eda["PJMW_MW"].mean(), 2))
print("Overall std MW:", round(eda["PJMW_MW"].std(), 2))
print("Minimum MW:", round(eda["PJMW_MW"].min(), 2))
print("Maximum MW:", round(eda["PJMW_MW"].max(), 2))
print("Weekday mean MW:", round(eda.loc[eda["Is_Weekend"] == 0, "PJMW_MW"].mean(), 2))
print("Weekend mean MW:", round(eda.loc[eda["Is_Weekend"] == 1, "PJMW_MW"].mean(), 2))
print("Holiday mean MW:", round(eda.loc[eda["Is_Holiday"], "PJMW_MW"].mean(), 2))
print("Non-holiday mean MW:", round(eda.loc[~eda["Is_Holiday"], "PJMW_MW"].mean(), 2))


### 4.1 Overall hourly consumption distribution

In [ ]:
# Distribution of hourly energy consumption
# Using Matplotlib directly avoids the Seaborn/Pandas compatibility issue.

plt.figure(figsize=(10, 5))

plt.hist(
    eda["PJMW_MW"].dropna(),
    bins=50,
    edgecolor="black"
)

plt.title("Distribution of Hourly Energy Consumption")
plt.xlabel("Energy Consumption (MW)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

### 4.2 Average consumption by hour of day

In [ ]:
hourly_avg = eda.groupby("Hour")["PJMW_MW"].mean()

plt.figure(figsize=(10, 5))
plt.plot(hourly_avg.index, hourly_avg.values, marker="o")
plt.title("Average Hourly Energy Consumption")
plt.xlabel("Hour of Day")
plt.ylabel("Average Consumption (MW)")
plt.xticks(range(24))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 Day-of-week analysis

In [ ]:
dow_avg = eda.groupby("DayOfWeek")["PJMW_MW"].mean()

plt.figure(figsize=(10, 5))
plt.bar(dow_avg.index, dow_avg.values)
plt.title("Average Energy Consumption by Day of Week")
plt.xlabel("Day of Week (0=Monday)")
plt.ylabel("Average Consumption (MW)")
plt.xticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.tight_layout()
plt.show()

### 4.4 Monthly / seasonal analysis

In [ ]:
monthly_avg = eda.groupby("Month")["PJMW_MW"].mean()

plt.figure(figsize=(10, 5))
plt.plot(monthly_avg.index, monthly_avg.values, marker="o")
plt.title("Average Energy Consumption by Month")
plt.xlabel("Month")
plt.ylabel("Average Consumption (MW)")
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

season_avg = eda.groupby("Season")["PJMW_MW"].mean().reindex(
    ["Winter", "Spring", "Summer", "Autumn"]
)

plt.figure(figsize=(9, 5))
plt.bar(season_avg.index, season_avg.values)
plt.title("Average Energy Consumption by Season")
plt.xlabel("Season")
plt.ylabel("Average Consumption (MW)")
plt.tight_layout()
plt.show()


### 4.5 Seasonal hourly pattern

In [ ]:
# Plot each season separately using Matplotlib
plt.figure(figsize=(12, 6))

for season in season_hour["Season"].unique():
    temp = season_hour[season_hour["Season"] == season]
    plt.plot(temp["Hour"], temp["PJMW_MW"], marker="o", label=season)

plt.title("Hourly Energy Consumption Pattern by Season")
plt.xlabel("Hour")
plt.ylabel("Average Consumption (MW)")
plt.xticks(range(24))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.6 Weekday vs weekend hourly pattern

In [ ]:
eda["Day_Type"] = np.where(
    eda["Is_Weekend"] == 1, "Weekend", "Weekday"
)

daytype_hour = (
    eda.groupby(["Day_Type", "Hour"])["PJMW_MW"]
       .mean()
       .reset_index()
)

plt.figure(figsize=(12, 6))
plt.figure(figsize=(12, 6))

for day_type in daytype_hour["Day_Type"].unique():
    temp = daytype_hour[daytype_hour["Day_Type"] == day_type]
    plt.plot(temp["Hour"], temp["PJMW_MW"], marker="o", label=day_type)

plt.title("Hourly Consumption: Weekday vs Weekend")
plt.xlabel("Hour")
plt.ylabel("Average Consumption (MW)")
plt.xticks(range(24))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.7 Holiday analysis

In [ ]:
holiday_avg = eda.groupby("Is_Holiday")["PJMW_MW"].mean()

print("Average consumption on non-holidays:", round(holiday_avg.get(False, np.nan), 2), "MW")
print("Average consumption on holidays:", round(holiday_avg.get(True, np.nan), 2), "MW")

plt.figure(figsize=(7, 5))
plt.bar(
    ["Non-Holiday", "Holiday"],
    [holiday_avg.get(False, np.nan), holiday_avg.get(True, np.nan)]
)
plt.title("Average Consumption: Holiday vs Non-Holiday")
plt.ylabel("Average Consumption (MW)")
plt.tight_layout()
plt.show()


### 4.8 Long-term annual trend

In [ ]:
yearly_avg = eda.groupby("Year")["PJMW_MW"].mean()

plt.figure(figsize=(12, 5))
plt.plot(yearly_avg.index, yearly_avg.values, marker="o")
plt.title("Long-Term Annual Energy Consumption Trend")
plt.xlabel("Year")
plt.ylabel("Average Consumption (MW)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 4.9 Daily average consumption trend

In [ ]:
daily_avg = eda["PJMW_MW"].resample("D").mean()

plt.figure(figsize=(15, 5))
plt.plot(daily_avg.index, daily_avg.values)
plt.title("Daily Average Energy Consumption")
plt.xlabel("Date")
plt.ylabel("Average Consumption (MW)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 4.10 Outlier analysis

In [ ]:
plt.boxplot(
    eda["PJMW_MW"].dropna(),
    vert=False
)

plt.title("Boxplot of Hourly Energy Consumption")
plt.xlabel("Energy Consumption (MW)")
plt.tight_layout()
plt.show()

low_outliers = int((eda["PJMW_MW"] < 1000).sum())
print("Values below 1000 MW:", low_outliers)

# The source ModelBuilding notebook treated values below 1000 MW as unrealistic
# and replaced them with the median. We report them here and apply the same
# treatment to the modeling series only.
model_ts = ts.copy()
safe_median = model_ts.median()

if low_outliers > 0:
    model_ts.loc[model_ts < 1000] = safe_median
    print(f"Treated {low_outliers} values below 1000 MW using median {safe_median:.2f} MW.")
else:
    print("No values below 1000 MW were found; no outlier replacement was required.")


### 4.11 Correlation analysis

In [ ]:
corr_df = eda[["PJMW_MW", "Hour", "DayOfWeek", "Month", "Year"]]

plt.figure(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


### 4.12 Time-series decomposition

In [ ]:
# Daily aggregation is used with weekly seasonality, matching the source EDA notebook.
daily_ts = model_ts.resample("D").mean().dropna()

decomposition = seasonal_decompose(
    daily_ts,
    model="additive",
    period=7
)

fig = decomposition.plot()
fig.set_size_inches(15, 10)
plt.tight_layout()
plt.show()


## 5. Key EDA findings

Based on the EDA , the main observations are:

1. **Strong hourly seasonality:** demand is lower during early morning hours and generally rises during the day.
2. **Weekday/weekend effect:** average weekday consumption is higher than weekend consumption.
3. **Holiday effect:** holiday consumption is somewhat different from normal days.
4. **Seasonal differences:** consumption varies across winter, spring, summer and autumn.
5. **Seasonal hourly patterns:** the hourly profile changes with season.
6. **Long-term fluctuations:** annual consumption does not follow a simple linear trend.
7. **Peak demand:** the dataset contains high-demand periods, with the source data reaching approximately 9,594 MW.
8. **Data quality:** duplicate/DST-related timestamps and missing hourly observations require time-series-specific handling.

These observations motivate lag-based and calendar/time features for the machine-learning models, while also explaining why seasonal-naive baselines are important.

## 6. Feature engineering

In [ ]:
# Work from the cleaned/imputed modeling series.
model_ts = model_ts.sort_index()

# Calendar variables
feature_df = model_ts.to_frame("PJMW_MW").copy()
feature_df["hour"] = feature_df.index.hour
feature_df["dow"] = feature_df.index.dayofweek
feature_df["month"] = feature_df.index.month
feature_df["year"] = feature_df.index.year
feature_df["doy"] = feature_df.index.dayofyear
feature_df["is_weekend"] = (feature_df["dow"] >= 5).astype(int)

# Lag features capturing recent, daily and weekly behavior.
for lag in [1, 24, 48, 168, 336]:
    feature_df[f"lag_{lag}"] = feature_df["PJMW_MW"].shift(lag)

# Rolling statistics are shifted by one hour so that the current target is not included.
for window in [24, 168, 720]:
    feature_df[f"roll_mean_{window}"] = (
        feature_df["PJMW_MW"].shift(1).rolling(window).mean()
    )
    feature_df[f"roll_std_{window}"] = (
        feature_df["PJMW_MW"].shift(1).rolling(window).std()
    )

# Cyclical encodings preserve the circular nature of hour/day/month.
feature_df["hour_sin"] = np.sin(2 * np.pi * feature_df["hour"] / 24)
feature_df["hour_cos"] = np.cos(2 * np.pi * feature_df["hour"] / 24)
feature_df["dow_sin"] = np.sin(2 * np.pi * feature_df["dow"] / 7)
feature_df["dow_cos"] = np.cos(2 * np.pi * feature_df["dow"] / 7)
feature_df["month_sin"] = np.sin(2 * np.pi * feature_df["month"] / 12)
feature_df["month_cos"] = np.cos(2 * np.pi * feature_df["month"] / 12)

# Holiday indicator.
feature_df["is_holiday"] = [
    int(d.date() in holiday_dates) for d in feature_df.index
]

feature_df = feature_df.dropna()

print("Feature-engineered shape:", feature_df.shape)
display(feature_df.head())


## 7. Time-based train/test split

For time-series forecasting, a random train/test split is inappropriate because it can allow future observations to influence the training data.

Following the project requirement, the **last 365 days / last year** are used as the held-out test period.

In [ ]:
# Use the last 365 days as the test set.
test_start = feature_df.index.max() - pd.DateOffset(days=365)

train_df = feature_df[feature_df.index < test_start].copy()
test_df = feature_df[feature_df.index >= test_start].copy()

target = "PJMW_MW"

feature_columns = [
    c for c in feature_df.columns
    if c != target
]

X_train = train_df[feature_columns]
y_train = train_df[target]

X_test = test_df[feature_columns]
y_test = test_df[target]

print("Train period:", train_df.index.min(), "to", train_df.index.max())
print("Test period :", test_df.index.min(), "to", test_df.index.max())
print("Training rows:", len(train_df))
print("Testing rows :", len(test_df))
print("Testing hours (approximately one year):", 365 * 24)


## 8. Five-model forecasting comparison

The five approaches are:

1. **Previous-Day Seasonal Naive (24h)** – predicts each hour using the corresponding value 24 hours earlier.
2. **Previous-Week Seasonal Naive (168h)** – predicts each hour using the same hour seven days earlier.
3. **HistGradientBoosting** – non-linear tree-based ensemble model.
4. **Random Forest** – tree-based ensemble model.
5. **XGBoost** – gradient-boosting benchmark.

The two seasonal-naive models are deliberately included as strong baselines. A complex model should only be selected if it improves the forecasting error on the held-out future period.

In [ ]:
# ---------------------------------------------------------
# Recursive feature builder for ML forecasting
# ---------------------------------------------------------
def row_features(history_values, dt, holiday_date_set):
    vals = np.asarray(history_values, dtype=float)

    result = {}

    # Historical lag features.
    for lag in [1, 24, 48, 168, 336]:
        result[f"lag_{lag}"] = vals[-lag]

    # Rolling statistics from historical values only.
    for window in [24, 168, 720]:
        arr = vals[-window:]
        result[f"roll_mean_{window}"] = arr.mean()
        result[f"roll_std_{window}"] = arr.std()

    # Calendar features.
    h = dt.hour
    dow = dt.dayofweek
    month = dt.month
    doy = dt.dayofyear

    result["hour"] = h
    result["dow"] = dow
    result["month"] = month
    result["year"] = dt.year
    result["doy"] = doy
    result["is_weekend"] = int(dow >= 5)

    # Cyclical encodings.
    result["hour_sin"] = np.sin(2 * np.pi * h / 24)
    result["hour_cos"] = np.cos(2 * np.pi * h / 24)
    result["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    result["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    result["month_sin"] = np.sin(2 * np.pi * month / 12)
    result["month_cos"] = np.cos(2 * np.pi * month / 12)

    result["is_holiday"] = int(dt.date() in holiday_date_set)

    return pd.DataFrame([result], index=[dt])[feature_columns]


def recursive_forecast(model, history_series, forecast_index, holiday_date_set):
    history = list(np.asarray(history_series, dtype=float))
    predictions = []

    for dt in forecast_index:
        X_row = row_features(history, dt, holiday_date_set)
        pred = float(model.predict(X_row)[0])
        predictions.append(pred)
        history.append(pred)

    return pd.Series(predictions, index=forecast_index, name="Prediction")


def seasonal_naive_forecast(history_series, forecast_index, seasonal_lag):
    history = list(np.asarray(history_series, dtype=float))
    predictions = []

    for _ in forecast_index:
        pred = history[-seasonal_lag]
        predictions.append(float(pred))
        history.append(float(pred))

    return pd.Series(predictions, index=forecast_index, name="Prediction")


### 8.1 Model 1 – Previous-Day Seasonal Naive (24h)

In [ ]:
seasonal_naive_24 = seasonal_naive_forecast(
    train_df[target],
    test_df.index,
    seasonal_lag=24
)

print("Model 1 completed:", len(seasonal_naive_24), "predictions")


### 8.2 Model 2 – Previous-Week Seasonal Naive (168h)

In [ ]:
seasonal_naive_168 = seasonal_naive_forecast(
    train_df[target],
    test_df.index,
    seasonal_lag=168
)

print("Model 2 completed:", len(seasonal_naive_168), "predictions")


### 8.3 Model 3 – HistGradientBoosting

In [ ]:
hgb_model = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_iter=500,
    max_leaf_nodes=31,
    max_depth=None,
    l2_regularization=1.0,
    random_state=RANDOM_STATE
)

hgb_model.fit(X_train.astype(np.float32), y_train.astype(np.float32))

hgb_test = recursive_forecast(
    hgb_model,
    train_df[target],
    test_df.index,
    holiday_dates
)

print("Model 3 completed:", len(hgb_test), "predictions")


### 8.4 Model 4 – Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model.fit(X_train.astype(np.float32), y_train.astype(np.float32))

rf_test = recursive_forecast(
    rf_model,
    train_df[target],
    test_df.index,
    holiday_dates
)

print("Model 4 completed:", len(rf_test), "predictions")


### 8.5 Model 5 – XGBoost

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_model.fit(X_train.astype(np.float32), y_train.astype(np.float32))

xgb_test = recursive_forecast(
    xgb_model,
    train_df[target],
    test_df.index,
    holiday_dates
)

print("Model 5 completed:", len(xgb_test), "predictions")


## 9. Model evaluation and selection

All five models are evaluated against the same chronological test period.

- **MAE:** average absolute forecasting error in MW; lower is better.
- **RMSE:** penalizes larger errors more strongly; lower is better.
- **MAPE:** average absolute percentage error; lower is better.

The final selection rule follows the source five-model notebook: **select the model with the lowest test-set MAE**, while using RMSE and MAPE for confirmation.

In [ ]:
def mape_percent(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    safe_actual = np.where(actual == 0, np.nan, actual)

    return np.nanmean(
        np.abs((actual - predicted) / safe_actual)
    ) * 100


predictions_dict = {
    "Previous-Day Seasonal Naive (24h)": seasonal_naive_24,
    "Previous-Week Seasonal Naive (168h)": seasonal_naive_168,
    "HistGradientBoosting": hgb_test,
    "Random Forest": rf_test,
    "XGBoost": xgb_test
}

results = []

for model_name, predictions in predictions_dict.items():
    mae = mean_absolute_error(test_df[target], predictions)
    rmse = np.sqrt(mean_squared_error(test_df[target], predictions))
    mape = mape_percent(test_df[target], predictions)

    results.append({
        "Model": model_name,
        "MAE_MW": mae,
        "RMSE_MW": rmse,
        "MAPE_Percent": mape
    })

model_comparison = (
    pd.DataFrame(results)
      .sort_values("MAE_MW")
      .reset_index(drop=True)
)


model_comparison_display = model_comparison.copy()

model_comparison_display["MAE_MW"] = model_comparison_display["MAE_MW"].round(3)
model_comparison_display["RMSE_MW"] = model_comparison_display["RMSE_MW"].round(3)
model_comparison_display["MAPE_Percent"] = model_comparison_display["MAPE_Percent"].round(3)

display(model_comparison_display)

# Select the best model based on the lowest MAE
best_model_name = model_comparison.loc[0, "Model"]
best_row = model_comparison.iloc[0]

print("\nBest Model Selected:")
print(best_model_name)

print(f"MAE  : {best_row['MAE_MW']:.3f} MW")
print(f"RMSE : {best_row['RMSE_MW']:.3f} MW")
print(f"MAPE : {best_row['MAPE_Percent']:.3f}%")

best_model_name = model_comparison.loc[0, "Model"]
best_row = model_comparison.iloc[0]

print("\nSelected model based on lowest MAE:", best_model_name)
print(f"MAE  : {best_row['MAE_MW']:.3f} MW")
print(f"RMSE : {best_row['RMSE_MW']:.3f} MW")
print(f"MAPE : {best_row['MAPE_Percent']:.3f}%")


## 10. Five-model comparison visualizations

In [ ]:
plt.figure(figsize=(11, 6))
plt.bar(model_comparison["Model"], model_comparison["MAE_MW"])
plt.title("Five-Model Comparison – MAE")
plt.xlabel("Model")
plt.ylabel("MAE (MW)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 6))
plt.bar(model_comparison["Model"], model_comparison["RMSE_MW"])
plt.title("Five-Model Comparison – RMSE")
plt.xlabel("Model")
plt.ylabel("RMSE (MW)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 6))
plt.bar(model_comparison["Model"], model_comparison["MAPE_Percent"])
plt.title("Five-Model Comparison – MAPE")
plt.xlabel("Model")
plt.ylabel("MAPE (%)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 11. Selected model – Actual vs Predicted

The selected model is evaluated visually against the held-out test period.

In [ ]:
best_predictions = predictions_dict[best_model_name]

plt.figure(figsize=(14, 5))
plt.plot(test_df.index, test_df[target].values, label="Actual")
plt.plot(
    best_predictions.index,
    best_predictions.values,
    label=f"Predicted – {best_model_name}"
)
plt.title(f"Actual vs Predicted Energy Consumption – {best_model_name}")
plt.xlabel("Datetime")
plt.ylabel("Energy Consumption (MW)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 12. Final 30-day hourly forecast

The selected model is now used to forecast the next 30 days, giving **720 hourly predictions**.

For the selected Previous-Day Seasonal Naive model, each future hour is predicted from the corresponding hour 24 hours earlier, recursively using the newly generated predictions once the forecast moves beyond the first day.

In [ ]:
# Future 30-day hourly index
future_index = pd.date_range(
    model_ts.index.max() + pd.Timedelta(hours=1),
    periods=FORECAST_HOURS,
    freq="h"
)

if best_model_name == "Previous-Day Seasonal Naive (24h)":
    final_forecast = seasonal_naive_forecast(
        model_ts,
        future_index,
        seasonal_lag=24
    )

elif best_model_name == "Previous-Week Seasonal Naive (168h)":
    final_forecast = seasonal_naive_forecast(
        model_ts,
        future_index,
        seasonal_lag=168
    )

elif best_model_name == "HistGradientBoosting":
    final_forecast = recursive_forecast(
        hgb_model,
        model_ts,
        future_index,
        holiday_dates
    )

elif best_model_name == "Random Forest":
    final_forecast = recursive_forecast(
        rf_model,
        model_ts,
        future_index,
        holiday_dates
    )

elif best_model_name == "XGBoost":
    final_forecast = recursive_forecast(
        xgb_model,
        model_ts,
        future_index,
        holiday_dates
    )

else:
    raise ValueError("Unknown selected model.")

print("Final selected model:", best_model_name)
print("Forecast start:", final_forecast.index.min())
print("Forecast end:", final_forecast.index.max())
print("Number of hourly predictions:", len(final_forecast))
print("Peak forecast:", round(final_forecast.max(), 2), "MW")
print("Minimum forecast:", round(final_forecast.min(), 2), "MW")

display(final_forecast.head(24).to_frame("Forecast_MW"))


## 13. Export hourly forecast and daily summary

In [ ]:
forecast_30d_final = pd.DataFrame({
    "Datetime": final_forecast.index,
    "Forecast_MW": final_forecast.values
})

forecast_30d_final.to_csv(
    "PJMW_Final_30_Day_Forecast.csv",
    index=False
)

forecast_summary = forecast_30d_final.copy()
forecast_summary["Forecast_Day"] = (
    np.arange(len(forecast_summary)) // 24
) + 1

daily_forecast_summary = (
    forecast_summary
    .groupby("Forecast_Day")
    .agg(
        Start_Datetime=("Datetime", "min"),
        End_Datetime=("Datetime", "max"),
        Average_Forecast_MW=("Forecast_MW", "mean"),
        Minimum_Forecast_MW=("Forecast_MW", "min"),
        Maximum_Forecast_MW=("Forecast_MW", "max")
    )
    .reset_index()
)

numeric_cols = [
    "Average_Forecast_MW",
    "Minimum_Forecast_MW",
    "Maximum_Forecast_MW"
]

daily_forecast_summary[numeric_cols] = (
    daily_forecast_summary[numeric_cols].round(2)
)

daily_forecast_summary.to_csv(
    "PJMW_Final_30_Day_Daily_Summary.csv",
    index=False
)

print("Exported hourly forecast:", len(forecast_30d_final), "rows")
print("Exported daily summary:", len(daily_forecast_summary), "days")

display(forecast_30d_final.head())
display(daily_forecast_summary.head())


## 14. Final 30-day forecast visualization

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(final_forecast.index, final_forecast.values)
plt.title(
    f"Next 30 Days Hourly Energy Consumption Forecast – {best_model_name}"
)
plt.xlabel("Datetime")
plt.ylabel("Forecast Energy Consumption (MW)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 15. Final conclusion

Five forecasting approaches were evaluated on the same chronological last-year test set. The models were compared using MAE, RMSE and MAPE.

The model with the lowest test-set MAE was selected for the final forecasting stage. The selected model is then used to generate 720 hourly predictions for the next 30 days.